# Preprocessing 

the purpose of this notebook is to evaluate ways to handle missing values, figure out what type of time series forecasting technique I want to use. 

#### Target Variable
At first I wanted to predict nitrogen dioxide levels, but given that the majority of the nitrogen dioxide data points are missing, I decided to change my target variable to Sulfur Dioxide since majority of it's data points aren't missing. 

#### Univariate or Multivariate time series forecasting?
Univariate is the most simplistic technique for time series forecasting especially in this sense with the amount of missing data points present. However, given other variables such as other pollutants and metrics, it may be worth looking at multivariate techniques, especially if these other variables have a correlation with sulfur dioxide. So, I decided that I want to do both univariate and multivariate time series forecasting techniques to not only see which method serves the best outcome, but to explore with missing values and multivariate time series techniques out of personal curiousity.

In [13]:
import pandas as pd
import numpy as np
import os

In [14]:
df = pd.read_csv("/Users/Owner/cmse492_project/data/processed/cleaned_data.csv")

df.head()

,O3,NO,NO2,NOx,SO2,CO,PM10,PM2_5,WindDir,WindSpeed,Temp,datetime
0,48.90907,0.25111,5.00001,5.38503,1.65808,0.089291,8.350,4.788,5.8,4.3,1.8,2021-01-01 01:00:00
1,51.66347,0.20926,4.48721,4.80807,1.18435,0.070492,5.125,2.642,8.0,4.3,1.8,2021-01-01 02:00:00
2,54.43647,0.21972,4.03844,4.37534,1.24356,0.077542,5.075,2.359,18.2,4.8,2.1,2021-01-01 03:00:00
3,53.59898,0.21972,3.26915,3.60605,0.35531,0.059919,4.075,1.910,21.5,5.0,2.2,2021-01-01 04:00:00
4,52.37067,0.15694,3.94238,4.18302,0.47374,0.052870,4.150,1.816,22.5,5.0,2.3,2021-01-01 05:00:00


In [15]:
df = df[df['datetime'] >= "2024-01-01"].copy()

df.head()

,O3,NO,NO2,NOx,SO2,CO,PM10,PM2_5,WindDir,WindSpeed,Temp,datetime
26280,59.67143,0.36182,5.04597,5.60076,0.30242,0.036575,7.125,3.892,284.2,8.1,4.7,2024-01-01 01:00:00
26281,61.86670,0.40705,4.07774,4.70187,0.10081,0.052830,5.500,2.972,284.2,8.2,4.6,2024-01-01 02:00:00
26282,59.92089,0.44097,4.87275,5.54890,0.30242,0.039622,4.675,2.618,275.2,7.3,4.4,2024-01-01 03:00:00
26283,61.01853,0.40705,3.31714,3.94128,0.15121,0.039622,5.675,2.618,270.8,7.2,4.3,2024-01-01 04:00:00
26284,65.75832,0.13568,2.33304,2.54109,0.37803,0.039622,3.950,1.934,266.2,7.8,4.6,2024-01-01 05:00:00


In [16]:
#making sure that the date-time values are sorted in order
df = df.sort_values("datetime").reset_index(drop=True)


In [17]:
#columns other than date time
columns = ["O3", "NO", "NO2", "NOx", "SO2", "CO", "PM10", "PM2_5", "WindDir", "WindSpeed", "Temp"]

a_columns = [c for c in columns if c.lower() in df.columns]

In [18]:
df.head()

,O3,NO,NO2,NOx,SO2,CO,PM10,PM2_5,WindDir,WindSpeed,Temp,datetime
0,NaN,NaN,NaN,NaN,0.30242,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 00:15:00
1,NaN,NaN,NaN,NaN,0.30242,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 00:30:00
2,NaN,NaN,NaN,NaN,0.30242,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 00:45:00
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 01:00:00
4,59.67143,0.36182,5.04597,5.60076,0.30242,0.036575,7.125,3.892,284.2,8.1,4.7,2024-01-01 01:00:00


In [19]:
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
print(df["datetime"].dtype)

datetime64[ns]


### Univariate Preprocessing
for this method, we are trying to predict future SO2 values by only comparing/evaluating it with the date time column. We are trying to evaluate any patterns and oddities in the SO2 values vs datetime. for this, we are going to create new columns, 

In [20]:
df_U = df[['datetime', 'SO2']].copy()

# Drop rows with missing SO2
df_U = df_U.dropna(subset=["SO2"])

print("shape of the univariate data set:", df_U.shape)


def add_time_features(df):
    """
    this fuction creates the necessary time columns needed for
    Univariate time series analysis.

    I also added cylical encoding which helps with the cylical 
    nature of time, i.e december is closer to january than to august.
    """
    df["hour"] = df["datetime"].dt.hour
    df["dayofweek"] = df["datetime"].dt.dayofweek
    
    # Cyclical encoding
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

    return df


def add_lag_features(df, target="SO2", lags=[1, 6, 12, 24]):
    """
    This function adds lag features which uses past values to 
    predict future values by finding the difference between past
    and present.
    """
    for l in lags:
        df[f"{target}_lag_{l}"] = df[target].shift(l)
    return df


df_U = add_time_features(df_U)
df_U = add_lag_features(df_U, target="SO2")
df_U = df_U.dropna().reset_index(drop=True)

print("the shape of the univariate data set after feature engineering", df_U.shape)

shape of the univariate data set: (45010, 2)
the shape of the univariate data set after feature engineering (44986, 12)


### Multivariate Preprocessing 
here is where we are doing the preprocessing for multivariate timeseries forecasting. the difference here is that we are adding the other variables into the mix, and filling the missing values using the median values.

In [21]:
df_M = df.copy()

# median imputation technique
for c in a_columns:
    df_M[c] = pd.to_numeric(df_M[c], errors="coerce")
    df_M[p] = df_M[c].fillna(df_M[c].median())

df_M = add_time_features(df_M)
df_M = add_lag_features(df_M, target="SO2")

df_M = df_M.dropna().reset_index(drop=True)

print("Option B after feature engineering:", df_M.shape)

Option B after feature engineering: (5024, 22)


In [22]:
# here is saving these new data sets to the data directory
processed_dir_U = "/Users/Owner/cmse492_project/data/processed/Univariate_data"
processed_dir_M = "/Users/Owner/cmse492_project/data/processed/Multivariate_data"
outpath_U = os.path.join(processed_dir_U, "Univariate_df.csv")
df_U.to_csv(outpath_U, index=False)
outpath_M = os.path.join(processed_dir_M, "Multivariate_df.csv")
df_M.to_csv(outpath_M, index=False)

print("Saved to:", outpath_U, " and ", outpath_M)

Saved to: /Users/Owner/cmse492_project/data/processed/Univariate_data\Univariate_df.csv  and  /Users/Owner/cmse492_project/data/processed/Multivariate_data\Multivariate_df.csv


### Test-Train split
now it is the last step before model selection and hyperparameter tuning, here I am splitting both the univariate and multivariate datasets into test and training sets. These will be saved in their respective folders in the data section, ready for model fitting. In the code below, I create a test-train split function myself since I want to make sure I am splitting the first 80% of the data for the training set and having the more recent, 20% of the data to be my test set. This is to ensure that the split isn't random since this is time series data.

In [23]:
def time_series_split(df, test_size=0.2):
    n = len(df)
    split_idx = int(n * (1 - test_size))
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]
    return train, test

train_U, test_U = time_series_split(df_U)
train_M, test_M = time_series_split(df_M)

print(len(train_U), len(test_U))
print(len(train_M), len(test_M))

35988 8998
4019 1005


In [24]:
outpath_U_train = os.path.join(processed_dir_U, "Univariate_train_U.csv")
train_U.to_csv(outpath_U_train, index=False)
outpath_U_test = os.path.join(processed_dir_U, "Univariate_test_U.csv")
test_U.to_csv(outpath_U_test, index=False)

outpath_M_train = os.path.join(processed_dir_M, "Multivariate_train_M.csv")
train_M.to_csv(outpath_M_train, index=False)
outpath_M_test = os.path.join(processed_dir_M, "Multivariate_test_M.csv")
test_M.to_csv(outpath_M_test, index=False)